# PMA 데이터 전처리 파이프라인

raw 데이터부터 학습 입력(`WSISurvivalDataset`)까지 만드는 전체 과정을 순서대로 실행하는 노트북.
각 단계는 실제 저장소의 CLI 스크립트를 그대로 호출한다(노트북 전용 로직은 없음) — 실행 순서와
단계별 입출력은 `README.md`와 동일하다.

커널은 프로젝트 루트(`config.py`가 있는 위치)에서 띄워야 한다.


## 전체 구조

```
raw WSI(.svs)              raw clinical.tsv           raw RNA tsv
     │                           │                          │
     ▼                           ▼                          │
[A] wsi_preprocess.py      [B] extract_os_labels.py         │
     │                           │                          │
     │                           ▼                          │
     │                     os_labels_{tcga,cptac}.csv        │
     │                           │                          │
     │                           └──────────┬───────────────┘
     │                                      ▼
     │                            [C] extract_rna_clinical.py
     │                                      │
     │                    rna_{tcga,cptac}.csv, clinical_{tcga,cptac}.csv
     │                                      │
     └──────────────────┬───────────────────┘
                         ▼
              [D] select_rnaseq_genes.py
                         │
        rna_gene_selection_intersection/selected_genes_top_1500.csv (INT1500)
                         │
                         ▼
              data/dataset.py::WSISurvivalDataset  ← 학습(train.py)이 이걸 씀
```


In [ ]:
import sys
from pathlib import Path

assert Path("config.py").exists(), "프로젝트 루트에서 커널을 띄워야 합니다."
sys.path.insert(0, ".")


## [A] WSI 타일링 + feature 추출

다른 단계와 독립적이라 가장 먼저(또는 B·C와 병렬로) 실행 가능. 가장 오래 걸리는 단계.

- 입력: `data/tcga_paad_wsi/*.svs`, `data/cptac_pda_wsi/*.svs`
- 출력: `data/patches_{tcga,cptac}/tiles/<slide_id>/*.jpg` + `features.pt`(기본 backbone=resnet50) + `slide_index_task*.csv`
- `--tiles-only`를 주면 타일링만 하고 feature 추출은 생략(다른 backbone으로 따로 뽑고 싶을 때)


In [ ]:
!python -m data.wsi_preprocess --dataset tcga
!python -m data.wsi_preprocess --dataset cptac

## [B] 생존 라벨(OS) 추출

[A]와 독립적.

- 입력: `data/raw/{TCGA,CPTAC}_clinic/clinical.tsv`
- 출력: `data/os_labels_{tcga,cptac}.csv`


In [ ]:
!python -m data.extract_os_labels

## [C] RNA + clinical 추출

**[B]가 먼저 끝나야 함** — OS 라벨과 inner join하기 때문.

- 입력: `data/raw/{TCGA,CPTAC}_RNA/<file_uuid>/*.tsv`, `data/raw/{TCGA,CPTAC}_clinic/clinical.tsv`, `data/os_labels_{tcga,cptac}.csv`([B] 산출물)
- 출력: `data/rna_{tcga,cptac}.csv`, `data/clinical_{tcga,cptac}.csv`, `data/common_genes.csv`


In [ ]:
!python -m data.extract_rna_clinical

## [D] INT1500 유전자 선정

**[A][B][C]가 전부 끝나야 함** — train split을 구하려면 patches([A])가, Cox 랭킹을 매기려면
RNA/OS 라벨([B][C])이 필요.

매개변수 없음. 내부적으로 다음을 순서대로 전부 수행한다:
1. TCGA train split만으로 Cox 순위 계산 → `data/rna_gene_selection_tcgaonly/`
2. CPTAC train split만으로 Cox 순위 계산 → `data/rna_gene_selection_cptaconly/`
3. 두 순위의 교집합(상위 1500개, INT1500) → `data/rna_gene_selection_intersection/selected_genes_top_1500.csv`

TCGA-only/CPTAC-only 순위(각자 자기 라벨만으로 독립 계산)가 겹치는 유전자만 쓰므로,
레퍼런스의 Stouffer 결합 방식과 달리 어느 방향으로 external 평가를 하든 leakage가 없다.


In [ ]:
!python -m data.select_rnaseq_genes


## 결과 확인

위 단계가 전부 끝나면 `WSISurvivalDataset`이 바로 로드 가능한 상태가 된다.


In [ ]:
from config import DataConfig
from data.dataset import WSISurvivalDataset

cfg = DataConfig()
train_ds = WSISurvivalDataset(cfg, dataset="cptac", split="train", with_clinical=True, with_rna=True)
print(f"train case 수: {len(train_ds)}")
sample = train_ds[0]
print(f"환자 1명당 슬라이드 수: {len(sample)}")
print(f"rna shape: {sample[0]['rna'].shape}")


## (선택) 다른 backbone feature

[A]는 기본 backbone(resnet50)으로만 feature를 뽑는다. UNI/UNI2-h 등 다른 backbone이 필요하면
[A] 완료 후 별도로:


In [ ]:
!python -m utils.extract_features --dataset tcga --backbone uni2
!python -m utils.extract_features --dataset cptac --backbone uni2
